# West Midlands EFDI — Spatial Weights and Neighbour Analysis (Python)

This notebook mirrors the R Markdown workflow so that you can run both and compare their results.

**Project folder:** `D:\Spatial Data Analysis`  
**Input:** `efdi_wm_clean.gpkg`  
**Target projected CRS:** EPSG:27700 — British National Grid (metres)

The analysis examines Queen-contiguity neighbours, polygon area, and whether LSOAs with many neighbours tend to be smaller or larger. It does not assume that urban or rural polygons must have a particular neighbour count; the actual geometry determines that.

## 1. Check the Python environment and packages

Run this cell first. It shows which Python interpreter your notebook is using and checks whether the required packages are available.

If a package is missing, the cell prints an installation command. It does **not** install anything or change your Conda environment automatically.

In [1]:
import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: c:\Users\ofosu\anaconda3\envs\dortmund-climate\python.exe
Python version: 3.12.16 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 20:13:05) [MSC v.1942 64 bit (AMD64)]


In [2]:
import os

# Locate the active Conda environment
prefix = os.environ.get("CONDA_PREFIX") or os.path.dirname(os.sys.executable)

# Locate Conda's Windows Library folder
lib = os.path.join(prefix, "Library")

# Configure native library paths
os.environ["PATH"] = (
    os.path.join(lib, "bin") + os.pathsep + os.environ["PATH"]
)

os.environ["GDAL_DATA"] = os.path.join(lib, "share", "gdal")
os.environ["PROJ_LIB"] = os.path.join(lib, "share", "proj")

print("GDAL_DATA:", os.environ["GDAL_DATA"])
print("PROJ_LIB:", os.environ["PROJ_LIB"])                # should show a path too

GDAL_DATA: C:\Users\ofosu\anaconda3\envs\dortmund-climate\Library\share\gdal
PROJ_LIB: C:\Users\ofosu\anaconda3\envs\dortmund-climate\Library\share\proj


In [3]:
required = ["geopandas", "libpysal", "esda", "pandas", "numpy", "matplotlib", "shapely"]

missing = []
for package in required:
    try:
        __import__(package)
        print(f"OK: {package}")
    except ImportError:
        missing.append(package)
        print(f"MISSING: {package}")

if missing:
    print("\nInstall missing packages in the environment shown above, for example:")
    print("conda install -c conda-forge geopandas pysal pandas numpy matplotlib")
else:
    print("\nAll required packages are available.")

c:\Users\ofosu\anaconda3\envs\dortmund-climate\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


OK: geopandas
OK: libpysal
OK: esda
OK: pandas
OK: numpy
OK: matplotlib
OK: shapely

All required packages are available.


## 2. Import the packages

- `geopandas` reads and analyses spatial vector data; it is the closest everyday equivalent to R's `sf`.
- `libpysal` creates spatial weights, including Queen contiguity.
- `esda` provides exploratory spatial statistics such as Moran's I.
- `pandas` manages attribute tables; `numpy` supports numerical operations.
- `matplotlib` creates plots.

In [4]:
import geopandas as gpd
import libpysal
from libpysal.weights import Queen
import esda
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("GeoPandas:", gpd.__version__)
print("libpysal:", libpysal.__version__)
print("esda:", esda.__version__)

GeoPandas: 1.2.0
libpysal: 4.15.1
esda: 2.10.0


## 3. Read the West Midlands dataset

The cleaned GeoPackage is `efdi_wm_clean.gpkg`. A GeoPackage can contain one or more layers. If the file contains multiple layers, inspect them and specify the correct layer in `read_file()`.

In [5]:
from pathlib import Path

# Main project folder
project_dir = Path(r"D:\Spatial Data Analysis")

# Cleaned GeoPackage
data_path = project_dir / "efdi_wm_clean.gpkg"

# Read the dataset
food = gpd.read_file(
    data_path,
    engine="fiona"
)

# Display dataset information
print(food.head())

CRSError: Invalid projection: EPSG:27700: (Internal Proj Error: proj_create: no database context specified)

In [10]:
from pathlib import Path

# Main project folder
project_dir = Path(r"D:\Spatial Data Analysis")

# Cleaned GeoPackage
data_path = project_dir / "efdi_wm_clean.gpkg"

# Check whether the file exists
print("File exists:", data_path.exists())

# Read the dataset using Fiona
food = gpd.read_file(data_path, engine="fiona")

# Display dataset information
print(food)

File exists: True


CRSError: Invalid projection: EPSG:27700: (Internal Proj Error: proj_create: no database context specified)

In [27]:
print(food.crs.to_epsg())

None


In [1]:
# 1. Check the CRS authority
food.crs.to_authority()

# 2. Try matching with a lower confidence threshold
food.crs.to_epsg(min_confidence=0)

# 3. Display the CRS name
food.crs.name

NameError: name 'food' is not defined

In [11]:
# List the attribute fields (columns)
food.columns.tolist()

['lsoa01cd',
 'lsoa01nm',
 'efdi',
 'efdi_decil',
 'imd_rank',
 'house_pric',
 'veg_cov',
 'res_mob',
 'geometry']

In [12]:
# Number of observations (rows)
len(food)

2365

In [13]:
# Preview the first five records
food.head()

,lsoa01cd,lsoa01nm,efdi,efdi_decil,imd_rank,house_pric,veg_cov,res_mob,geometry
0,E01009143,Birmingham 059B,38.30457,1,765,220000.0,0.810,0.084,"MULTIPOLYGON (((405342.998 286770.002, 405353...."
1,E01009145,Birmingham 060D,18.79994,6,10049,230747.5,0.924,0.143,"MULTIPOLYGON (((403486.646 287261.011, 403497...."
2,E01009146,Birmingham 059D,26.37110,3,3785,271875.0,0.512,0.201,"MULTIPOLYGON (((406107.117 286605.7, 406086.00..."
3,E01009147,Birmingham 060E,18.25160,6,2695,339150.0,0.849,0.172,"MULTIPOLYGON (((403193.486 286134.974, 403181...."
4,E01009151,Birmingham 053A,26.51394,3,4545,145500.0,0.446,0.082,"MULTIPOLYGON (((403690.406 287732.187, 403710...."


In [14]:
# Data types, non-null counts, and memory information
food.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 2365 entries, 0 to 2364
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   lsoa01cd    2365 non-null   str     
 1   lsoa01nm    2365 non-null   str     
 2   efdi        2365 non-null   float64 
 3   efdi_decil  2365 non-null   int64   
 4   imd_rank    2365 non-null   int64   
 5   house_pric  2334 non-null   float64 
 6   veg_cov     2320 non-null   float64 
 7   res_mob     2365 non-null   float64 
 8   geometry    2365 non-null   geometry
dtypes: float64(4), geometry(1), int64(2), str(2)
memory usage: 166.4 KB


In [31]:
food.loc[
    ~food.geometry.is_valid,
    ["lsoa01cd", "lsoa01nm"]
]

,lsoa01cd,lsoa01nm
196,E01009012,Birmingham 028C
230,E01009048,Birmingham 040D
399,E01009267,Birmingham 027B
1454,E01010409,Walsall 027D
1725,E01029397,Cannock Chase 013D


In [32]:
import shapely

invalid_geometries = food.loc[
    ~food.geometry.is_valid
].copy()

invalid_geometries["invalid_reason"] = (
    shapely.is_valid_reason(invalid_geometries.geometry)
)

invalid_geometries[
    ["lsoa01cd", "lsoa01nm", "invalid_reason"]
]

,lsoa01cd,lsoa01nm,invalid_reason
196,E01009012,Birmingham 028C,Ring Self-intersection[411153.6949 291646.954]
230,E01009048,Birmingham 040D,Ring Self-intersection[405889.7984 289807.168]
399,E01009267,Birmingham 027B,Ring Self-intersection[404743.3239 291514.2303]
1454,E01010409,Walsall 027D,Ring Self-intersection[397326.626 299015.9662]
1725,E01029397,Cannock Chase 013D,Ring Self-intersection[402336.0186 307718.5879]


In [33]:
food["geometry"] = shapely.make_valid(food.geometry)

food.geometry.is_valid.value_counts()

True    2365
Name: count, dtype: int64

In [37]:
import shapely

food["geometry"] = shapely.make_valid(food.geometry)

In [38]:
print("Total observations:", len(food))
print("Valid geometries:", food.geometry.is_valid.sum())
print("Invalid geometries:", (~food.geometry.is_valid).sum())
print("Geometry types:")
print(food.geom_type.value_counts())

Total observations: 2365
Valid geometries: 2365
Invalid geometries: 0
Geometry types:
MultiPolygon    2360
Polygon            5
Name: count, dtype: int64


## 4. Check the coordinate reference system (CRS)

Distance and area calculations must use a suitable projected CRS measured in metres. EPSG:27700 is the British National Grid, appropriate for this West Midlands analysis.

First inspect the CRS of the input. Then create a projected copy for metric calculations.

In [4]:
print("Original CRS:", food.crs)

if food.crs is None:
    raise ValueError(
        "The dataset has no CRS defined. Identify its true CRS before assigning or transforming one."
    )

food = food.to_crs(epsg=27700)
print("Analysis CRS:", food.crs)

NameError: name 'food' is not defined

## 5. Check geometry quality

Queen contiguity is based on polygon geometry: two polygons are neighbours if they share an edge or at least one corner. Invalid or empty geometries can affect spatial-weights construction.

This cell reports empty and invalid geometries. It creates a valid-geometry analysis copy using `make_valid()` where needed, while keeping the original `food` object unchanged for reference. Compare the row count before and after; do not silently discard records.

In [ ]:
print("Rows before geometry checks:", len(food))
print("Empty geometries:", int(food.geometry.is_empty.sum()))
print("Missing geometries:", int(food.geometry.isna().sum()))
print("Invalid geometries:", int((~food.geometry.is_valid).sum()))

food_analysis = food.loc[
    food.geometry.notna() & ~food.geometry.is_empty
].copy()

invalid_mask = ~food_analysis.geometry.is_valid
if invalid_mask.any():
    food_analysis.loc[invalid_mask, "geometry"] = (
        food_analysis.loc[invalid_mask, "geometry"].make_valid()
    )

print("Rows used for geometry analysis:", len(food_analysis))
print("Remaining invalid geometries:", int((~food_analysis.geometry.is_valid).sum()))

## 6. Create Queen-contiguity spatial weights

Queen contiguity defines two polygons as neighbours when they touch at an edge **or a corner**.

`Queen.from_dataframe()` builds a neighbour structure from the polygons. The weights object stores which observations are neighbours. We keep the weights binary here: a neighbour relationship is recorded as present, not given a distance-based strength.

**Important for comparison:** the R and Python weights must use the same polygons, the same Queen definition, and the same observation order/IDs. If your GeoDa `wm_queen.gal` was created from a different set or order of observations, the results may not match until they are aligned.

In [ ]:
# Use a stable LSOA identifier if it exists.
possible_id_fields = ["lsoa01cd", "LSOA11CD", "lsoa11cd", "LSOA_CODE", "lsoa_code"]
id_field = next((c for c in possible_id_fields if c in food_analysis.columns), None)

if id_field:
    ids = food_analysis[id_field].astype(str).tolist()
    if pd.Series(ids).duplicated().any():
        raise ValueError(f"{id_field} contains duplicate IDs; choose a unique identifier.")
    w_queen = Queen.from_dataframe(food_analysis, ids=ids)
    print("Weights IDs use:", id_field)
else:
    w_queen = Queen.from_dataframe(food_analysis)
    print("No recognised LSOA ID field found; weights use row-based IDs.")

print("Number of observations in weights:", w_queen.n)
print("Number of islands (zero neighbours):", len(w_queen.islands))

In [ ]:
# Make a table of the neighbour count for every observation.
# cardinalities maps each weights ID to its number of neighbours.
neighbour_counts = pd.Series(w_queen.cardinalities, name="neighbours")

food_analysis = food_analysis.copy()
if id_field:
    food_analysis["_weights_id"] = food_analysis[id_field].astype(str)
else:
    food_analysis["_weights_id"] = food_analysis.index

food_analysis["neighbours"] = (
    food_analysis["_weights_id"].map(neighbour_counts).fillna(0).astype(int)
)

food_analysis[["neighbours"]].describe()

## 7. Inspect the Queen weights

The summary below describes how many neighbours LSOAs have. The minimum can be zero if there are islands (polygons with no Queen neighbours). The maximum is the largest neighbour count in the dataset.

The frequency table answers: **How many LSOAs have 1 neighbour, 2 neighbours, 3 neighbours, and so on?**

In [ ]:
print("Minimum neighbours:", int(food_analysis["neighbours"].min()))
print("Maximum neighbours:", int(food_analysis["neighbours"].max()))
print("Mean neighbours:", food_analysis["neighbours"].mean())
print("Median neighbours:", food_analysis["neighbours"].median())
print("Islands:", int((food_analysis["neighbours"] == 0).sum()))

frequency = (
    food_analysis["neighbours"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_neighbours")
    .reset_index(name="number_of_lsoas")
)
frequency["percent"] = 100 * frequency["number_of_lsoas"] / len(food_analysis)
frequency

In [ ]:
ax = frequency.plot(
    x="number_of_neighbours",
    y="number_of_lsoas",
    kind="bar",
    legend=False,
    figsize=(10, 5)
)
ax.set_xlabel("Number of Queen neighbours per LSOA")
ax.set_ylabel("Number of LSOAs")
ax.set_title("Distribution of Queen-contiguity neighbour counts")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Calculate polygon area in square kilometres

Because the data are now in EPSG:27700, geometry area is measured in square metres. Dividing by 1,000,000 converts it to square kilometres.

This is the area of each LSOA polygon, not its population or built-up area.

In [ ]:
food_analysis["area_km2"] = food_analysis.geometry.area / 1_000_000

food_analysis[["area_km2", "neighbours"]].describe()

## 9. Identify LSOAs with the highest number of neighbours

This table ranks LSOAs by neighbour count. It also reports polygon area so that you can inspect whether the LSOAs with the most neighbours are small or large.

The LSOA identifier field is included when one was detected.

In [ ]:
sort_columns = ["neighbours", "area_km2"]
top20 = food_analysis.sort_values(
    sort_columns,
    ascending=[False, False]
).head(20)

display_columns = ([id_field] if id_field else []) + ["neighbours", "area_km2"]
top20[display_columns]

## 10. Map the neighbour counts

The choropleth shows the number of Queen neighbours for each LSOA. Darker or higher classes indicate a greater number of adjacent polygons.

This is a map of polygon adjacency, not a map of population density, urbanity, or food-desert severity.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
food_analysis.plot(
    column="neighbours",
    cmap="viridis",
    legend=True,
    linewidth=0.05,
    ax=ax,
    missing_kwds={"color": "lightgrey", "label": "Missing"}
)
ax.set_title("Queen-contiguity neighbours by LSOA — West Midlands")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 11. Map the LSOAs with the maximum neighbour count

This map highlights all LSOAs tied for the maximum neighbour count. It helps locate where the highest counts occur geographically.

A polygon may have many neighbours because of its boundary arrangement. Do not conclude that it is rural or urban from neighbour count alone; inspect its location, shape, area, and surrounding polygons.

In [ ]:
max_neighbours = food_analysis["neighbours"].max()
max_lsoas = food_analysis.loc[food_analysis["neighbours"] == max_neighbours]

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
food_analysis.plot(ax=ax, color="whitesmoke", edgecolor="lightgrey", linewidth=0.15)
max_lsoas.plot(ax=ax, color="red", edgecolor="black", linewidth=0.6)
ax.set_title(f"LSOAs with the maximum neighbour count ({max_neighbours})")
ax.set_axis_off()
plt.tight_layout()
plt.show()

print("Number of LSOAs tied at the maximum:", len(max_lsoas))

## 12. Compare neighbour count with polygon area

The scatterplot places polygon area on the x-axis and number of neighbours on the y-axis. Each point represents one LSOA.

Look for the overall pattern, but remember that a relationship may be weak and that polygon shape and boundary arrangement also affect neighbour count.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    food_analysis["area_km2"],
    food_analysis["neighbours"],
    alpha=0.5
)
ax.set_xlabel("LSOA polygon area (km²)")
ax.set_ylabel("Number of Queen neighbours")
ax.set_title("Neighbour count versus LSOA polygon area")
plt.tight_layout()
plt.show()

In [ ]:
# Spearman correlation is useful for assessing a monotonic association.
# It does not imply that area causes a particular neighbour count.
corr = food_analysis[["area_km2", "neighbours"]].corr(method="spearman").iloc[0, 1]
print("Spearman correlation:", corr)

## 13. Compare polygon area across neighbour-count groups

The boxplot compares area distributions for LSOAs with different neighbour counts. If there are many distinct counts, the chart can become crowded; you can focus on selected counts or group them into ranges.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
food_analysis.boxplot(
    column="area_km2",
    by="neighbours",
    ax=ax,
    grid=False,
    showfliers=False
)
ax.set_xlabel("Number of Queen neighbours")
ax.set_ylabel("LSOA area (km²)")
ax.set_title("LSOA area by number of Queen neighbours")
fig.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics for area at each neighbour count
area_by_neighbours = (
    food_analysis.groupby("neighbours")["area_km2"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)
area_by_neighbours

## 14. Optional: group neighbour counts into bands

This gives a simpler comparison if the exact neighbour-count categories are too numerous. The bands are descriptive groupings for this analysis, not official classifications.

In [ ]:
bins = [-1, 3, 6, 9, np.inf]
labels = ["0–3", "4–6", "7–9", "10+"]
food_analysis["neighbour_band"] = pd.cut(
    food_analysis["neighbours"],
    bins=bins,
    labels=labels
)

band_summary = (
    food_analysis.groupby("neighbour_band", observed=False)["area_km2"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)
band_summary

## 15. Check EFDI missing values and prepare the analysis sample

The EFDI field is a continuous index. A missing EFDI value is not zero. The following cell checks the field and counts missing observations.

For a later Moran's I comparison, the spatial weights must be rebuilt for exactly the same rows used in the EFDI analysis. Do not simply remove rows from the data while reusing weights created for the full dataset.

In [ ]:
if "efdi" not in food_analysis.columns:
    print("The column 'efdi' was not found. Check the exact field name:")
    print(food_analysis.columns.tolist())
else:
    print("EFDI missing values:", int(food_analysis["efdi"].isna().sum()))
    print("EFDI non-missing values:", int(food_analysis["efdi"].notna().sum()))
    print(food_analysis["efdi"].describe())

## 16. Optional: Global Moran's I for EFDI

Run this section only if the `efdi` field exists and you want to compare spatial autocorrelation in Python with the R `moran.test()` / `moran.mc()` workflow.

The weights are rebuilt after selecting non-missing EFDI records so that the data and weights have matching observations. This is a basic complete-case approach; for a rigorous comparison, ensure the R workflow uses the identical records and matching Queen-neighbour definition.

In [ ]:
if "efdi" in food_analysis.columns:
    food_efdi = food_analysis.loc[food_analysis["efdi"].notna()].copy()

    if id_field:
        efdi_ids = food_efdi[id_field].astype(str).tolist()
        w_efdi = Queen.from_dataframe(food_efdi, ids=efdi_ids)
    else:
        w_efdi = Queen.from_dataframe(food_efdi)

    w_efdi.transform = "R"
    y = food_efdi["efdi"].to_numpy(dtype=float)

    moran_global = esda.Moran(y, w_efdi, permutations=999)
    print("Global Moran's I:", moran_global.I)
    print("Expected I:", moran_global.EI)
    print("Permutation p-value:", moran_global.p_sim)
else:
    print("Skipping Moran's I: no 'efdi' column.")

## 17. Optional: Local Moran's I (LISA) for EFDI

Local Moran's I identifies local spatial association patterns. `q` codes the quadrant classification:
- 1 = High–High
- 2 = Low–High
- 3 = Low–Low
- 4 = High–Low

The significance value is permutation-based. Interpret clusters only alongside the significance threshold and the substantive meaning of EFDI.

In [ ]:
if "efdi" in food_analysis.columns:
    food_efdi = food_analysis.loc[food_analysis["efdi"].notna()].copy()

    if id_field:
        efdi_ids = food_efdi[id_field].astype(str).tolist()
        w_efdi = Queen.from_dataframe(food_efdi, ids=efdi_ids)
    else:
        w_efdi = Queen.from_dataframe(food_efdi)

    w_efdi.transform = "R"
    y = food_efdi["efdi"].to_numpy(dtype=float)

    lisa = esda.Moran_Local(y, w_efdi, permutations=999)
    food_efdi["local_I"] = lisa.Is
    food_efdi["p_sim"] = lisa.p_sim
    food_efdi["quadrant"] = lisa.q

    food_efdi[["efdi", "local_I", "p_sim", "quadrant"]].head()
else:
    print("Skipping Local Moran's I: no 'efdi' column.")

## 18. Save analysis outputs (optional)

This saves the analysis table as a new GeoPackage in the same project folder. It does not overwrite the original input file.

In [ ]:
output_path = project_dir / "efdi_wm_neighbour_analysis.gpkg"

food_analysis.to_file(
    output_path,
    layer="neighbour_analysis",
    driver="GPKG"
)

print("Saved:", output_path)

## 19. Comparison checklist: Python versus R

When you compare the notebook with the R Markdown output, check that:

1. Both read the same cleaned GeoPackage and use the same number of observations.
2. Both use EPSG:27700 for area and distance calculations.
3. Both use Queen contiguity (edge **or** corner), not Rook contiguity.
4. Both use the same geometry repair/exclusion decisions.
5. Both use the same LSOA identifiers and observation order.
6. Neighbour counts, minimum, maximum, mean, median, and island counts agree.
7. Polygon areas agree (allowing for tiny floating-point differences).
8. For Moran's I, both use the same non-missing EFDI rows, weights, row-standardisation, and permutation settings.

Small differences can arise from package versions or geometry handling. If results differ, compare the input rows, CRS, geometry validity, IDs, and weights before interpreting the statistics.